In [ ]:
%cd /kaggle/
!git clone https://github.com/oobabooga/text-generation-webui.git
%cd text-generation-webui
!git checkout 80cdbe4e09162caead35ba6212772896a4a43023
%cd ..

In [ ]:
import os

os.chdir("/kaggle/text-generation-webui/")

new_content = "--api --trust-remote-code"

with open("CMD_FLAGS.txt", "w") as file:
    file.write(new_content)

print("文件修改完成。")


In [ ]:
!conda create -n textgen python=3.11 -y

In [ ]:
!source /opt/conda/bin/activate textgen && pip3 install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121
!source /opt/conda/bin/activate textgen && conda install -y -c "nvidia/label/cuda-12.1.1" cuda-runtime

In [ ]:
%cd /kaggle/
%cd text-generation-webui/instruction-templates
content = r"""
instruction_template: |-
  {{ '<bos>' }}
  {% if messages[0]['role'] == 'system' %}
      {{ '' }}
  {% endif %}
  {% for message in messages %}
      {% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}
          {{ '' }}
      {% endif %}
      {% if message['role'] == 'assistant' %}
          {% set role = 'model' %}
      {% else %}
          {% set role = message['role'] %}
      {% endif %}
      {{ '<start_of_turn>' + role + '\n' + message['content'] | trim + '<end_of_turn>\n' }}
  {% endfor %}
"""

with open('gemma2.yaml', 'w') as file:
    file.write(content)
    
print("文件修改完成。")

In [ ]:
%cd /kaggle/
%cd text-generation-webui
!source /opt/conda/bin/activate textgen && pip install -r requirements.txt
!source /opt/conda/bin/activate textgen && pip install pydantic==2.8.2 pydantic-core==2.20.1 fastapi==0.112.4

In [ ]:
Ngrok_token = ""  # ngrok token
port1 = 7860
port2 = 5000
HOME_FOLDER = "/kaggle"

!pip install pyngrok==6.1.0  # Downgrade to a version that supports ngrok tunneling

from pyngrok import ngrok, conf
import gc

gc.collect()

if Ngrok_token:
    try:
        ngrok.set_auth_token(Ngrok_token)
        ngrok.kill()
        srv1 = ngrok.connect(port1)
        srv2 = ngrok.connect(port2)
        print(f"Ngrok Tunnel 1 is active at: {srv1.public_url}")
        print(f"Ngrok Tunnel 2 is active at: {srv2.public_url}")
        
        # Replace this line with the command to start the text-generation-webui
        !source /opt/conda/bin/activate textgen && python server.py --api --trust-remote-code
        
    except Exception as e:
        print(f"Error starting ngrok tunnel: {e}")
else:
    print('An ngrok token is required. You can get one on https://ngrok.com and paste it into the Ngrok_token field.')
